# 1. General Information

In this Notebook, we will explore and analyze data related to Life Expectancy across different countries and years. The primary focus will be on understanding the factors that influence life expectancy using factors that are in the data. We will start by visualizing the data to identify trends and patterns. Following this, we will apply statistical and machine learning models to predict life expectancy based on various indicators.

The goals of this notebook are:

* Data Exploration and Visualization: To examine the dataset, understand its structure, and identify any significant correlations or trends.
* Data Preprocessing: To clean and prepare the data for modeling, addressing any missing values or inconsistencies.
* Modeling Life Expectancy: To build predictive models that estimate life expectancy based on the provided features.
* Evaluation and Interpretation: To assess the performance of the models and interpret the results, drawing meaningful insights that can help inform public health strategies and policies.

By the end of this analysis, I hope to gain a deeper understanding of the key determinants of life expectancy and how these factors vary across different regions and periods.

# 2. Information about the dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
data = pd.read_csv('Life Expectancy Data.csv.xls')
data.info()

As we see we have some null values in several features. Also we can see, that several columns have unnecessary spaces in their names. Antoher factor is that the names of the columns are not normalized - we will use snake case. And we will also change the datatype of `Year` to string.

In [ ]:
data['Year'] = data['Year'].astype('object')

In [ ]:
def snake_case(df_index: pd.Index) -> pd.Index:
    """
    Converts the column or index names from a Pandas DataFrame to snake_case format.

    This function replaces spaces in the names with underscores ("_") 
    and converts all characters to lowercase, following the snake_case naming convention.

    :param df_index: 
        The Pandas Index object, typically from `data.columns` or `data.index`, 
        representing the names to be transformed.
    :return: 
        A Pandas Index object with names converted to snake_case.
    """
    return df_index.str.strip().str.lower().str.replace(r'\s{1,2}', '_', regex=True)


In [ ]:
data.columns = snake_case(data.columns)
data.head(2)

# 3. Splitting the data

In this part, we will simply split the data in train / validation / test dataset. However, first, we will try to find a good feature to stratify the data. After this step, we will no longer look at the test set. Stratifying the data is important beacuse using simple `train_test_split` can give us non population like results. For instance we can have a lot of countries that are developed and just few countries that are still developing which can result in a highly biased model.

## 3.1 Checking the Status Variable

Lets check if `status` variable is good for statifying the data, by visualing it.

In [ ]:
plt.figure(figsize=(8,6))
counts = data["status"].value_counts()
counts.plot(kind="bar");
plt.xlabel('');
plt.ylabel("Count");
plt.title("Record count by status");
plt.grid()

In [ ]:
plt.figure(figsize=(8,6))
sns.pointplot(data, y="status", x="life_expectancy");
plt.title("Life Expectancy by Status");
plt.grid()

In [ ]:
plt.figure(figsize=(12,6))
sns.scatterplot(data, x="adult_mortality",y="life_expectancy", hue="status")
plt.grid()

In [ ]:
from scipy.stats import f_oneway
developing = data[data['status']=="Developing"]["life_expectancy"]
developed = data[data['status']=="Developed"]["life_expectancy"]
f_oneway(developed, developing, nan_policy = 'omit')

Visualisation and Anova test shown us, that there's a big difference between counties that are developing and those caounties that are developed - which can be a good thing to stratify the data.

## 3.2 Stratifying the Data

Based on the plot that is shown above we can see that Countries differs from one another. The biggest issue by stratifying the data by `status` is that, the model will not be trained for each country, beacause sample data will not take countries into consideration. For experimentation purposes we will stratify this just by status.


In [ ]:
from sklearn.model_selection import train_test_split

strat_train_set, strat_test_set = train_test_split(
    data,
    test_size=0.30,
    stratify=data['status']
)

In [ ]:
data['status'].value_counts() / len(data) * 100

In [ ]:
strat_train_set['status'].value_counts() / len(strat_train_set) * 100

In [ ]:
strat_test_set['status'].value_counts() / len(strat_test_set) * 100

# 4. Exploratory Data Analysis

In this part we will:

* Check the quality of the data,

* Clean the data,

* Search for valuable correlation between the features,

* Prepare the data for modelling by building dedicated pipelines.


## 4.1 Data Cleansing

As we know from the second chapter, we have some features with missing data. In this subchapter we will try to do someting about it to increase the quality of the data.

In [ ]:
missing_value_cols = strat_train_set.columns[strat_train_set.isnull().any()].tolist()
strat_train_set[missing_value_cols].describe()

As we know from previous analysis, countries can vary from eachother, and by that it could be a bad idea to fill the missing values by metrics like `median` or `mean`. Instead, we will use `KNNInputer` which search for n number of closest neighbors and take the mean value from them to fill a missing value.

In [ ]:
from sklearn.impute import KNNImputer

inputer = KNNImputer(n_neighbors=7)
values = inputer.fit_transform(strat_train_set[missing_value_cols])
strat_train_set[missing_value_cols] = values
number_of_nans = strat_train_set.isnull().sum().sum()
print(f"Number of empty values in dataset: {number_of_nans}")


For keeping the train data as clean as we can, we will perform analysis on the copy of the train dataset.

In [ ]:
experimental_train_data = strat_train_set.copy()
experimental_train_data = experimental_train_data.reset_index()
experimental_train_data.drop('index',axis=1, inplace=True)

## 4.2 Data Visualisation and Statistical Analysis

In this part we will try to visualise the data to get better insights into the data. We will also try to use some statistic knowledge to try to extract what's the best in those features.

### 4.2.1 Dealing with Skewed data

In this section we will perform the analysis of descriptive statistics.

In [ ]:
desc = experimental_train_data.describe()
skew = experimental_train_data.select_dtypes(np.number).skew()
desc.loc["skew"] = skew
desc

As first glance we can see, that most of our data will be needing scalling, cause many columns are in different scale. The second thing we can notice, is that we have some really skewed data (both left and right sided) which may impact the result of the modelling. Let's see how they look like, and what can we do about them.

In [ ]:
skewed_cols = skew.index[np.abs(skew.values) > 1].to_list()

In [ ]:
fig, ax = plt.subplots(2,7, figsize=(25,10))
fig.tight_layout(pad=5.)
for i,col in enumerate(skewed_cols):
    row = i // 7
    col_idx = i % 7
    sns.histplot(data=experimental_train_data, x=experimental_train_data[col], ax=ax[row,col_idx], kde=True).grid(True)

For features that are skewed to the right, we can take `log` from them to make them look more normal like distributed. For left skewed data we can use many different transformations (like Box-cox, or taking the feature to the `n` power.) to also make them more normal distributed. We also can see several features with bimodal distribution, we will perform the `RBF` (radial basis function) to calculate the probability of each sample to the modes in each feature.

In [ ]:
log_skewed_features = skew.index[skew.values > 1].to_list()[:-2] # removing the bimodal features

fig, ax = plt.subplots(4,2, figsize=(15,20))
fig.suptitle("Right skewed data after transformation", fontsize=20)
for i, col in enumerate(log_skewed_features):
    row = i // 2
    col_idx = i % 2
    #data = strat_train_set[strat_train_set[col]>0]
    sns.histplot(np.log(experimental_train_data[col]), kde=True, ax = ax[row, col_idx]).grid(True)
    experimental_train_data[f"log_{col}"] = np.log(experimental_train_data[col]).replace([np.inf, -np.inf], 0)
plt.tight_layout(rect=[0, 0, 1, 0.93])

Lets see the results of the transformation:

In [ ]:
log_trans_features = [col for col in experimental_train_data.columns if 'log_' in col]
print("Skewness before transformation:\n")
print(experimental_train_data[log_skewed_features].skew(),"\n")

print("Skewness after transformation:\n")
print(experimental_train_data[log_trans_features].skew())

As we see taking `log` from right skewed data, can significantly lower the skewness in our data and (probably) increase the correlation with target variable (we will check it a little bit later). Now - lets focus on left skewed data.

In [ ]:
left_skewed_features = skew.index[skew.values < -1].to_list()[:-1] #Removing the bimodal feature

fig, ax = plt.subplots(2,2,figsize=(20,16))
fig.suptitle("Left skewed data before transformation", fontsize=20, );
fig.tight_layout()
for i, col in enumerate(left_skewed_features):
    row = i // 2
    col_idx = i % 2
    sns.histplot(experimental_train_data[col], kde=True, ax=ax[row, col_idx]).grid(True)
ax[1,1].axis('off')

In [ ]:
fig, ax = plt.subplots(2,2,figsize=(20,16))
fig.suptitle("Left skewed data after transformation", fontsize=20);
fig.tight_layout()
for i, col in enumerate(left_skewed_features):
    row = i // 2
    col_idx = i % 2
    sns.histplot(experimental_train_data[col]**3, ax=ax[row, col_idx], kde=True).grid(True) #Taking the feature to the third power.
    experimental_train_data[f"pow3_{col}"] = experimental_train_data[col]**3
ax[1,1].axis('off')

In [ ]:
power_cols = [col for col in experimental_train_data.columns if 'pow3' in col]
print("Skewness before transformation: \n")
print(experimental_train_data[left_skewed_features].skew(), "\n")
print("Skewness after transformation: \n")
print(experimental_train_data[power_cols].skew())

Taking the features to the power of 3 seems like a good idea to reduce the skewness in our data, however it's only working with left skewed data. Now let's focus on the binominal features. We will build a custom transformer that will measure the probability of belonging to a particular cluster defined by `KMeans` algorithm.

In [ ]:
binom_features = ['income_composition_of_resources','thinness_5-9_years','thinness_1-19_years']
fig, ax = plt.subplots(1,3, figsize=(20,10))
for i, col in enumerate(binom_features):
    sns.histplot(experimental_train_data[col], ax=ax[i], kde=False, stat='density')
    sns.kdeplot(experimental_train_data[col], ax=ax[i], color='crimson')
    ax[i].grid(True)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=3, gamma = 0.1, random_state = None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state
        
    def fit(self, X, y=None, sample_weight = None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma = self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"proba_{i}_cluster" for i in range (self.n_clusters)]



In [ ]:
CS = ClusterSimilarity(n_clusters=3)
preds = CS.fit_transform(experimental_train_data[['thinness_5-9_years','thinness_1-19_years']]).round(3)
CS_data = pd.DataFrame(
    preds,
    columns=CS.get_feature_names_out()
)
CS_data


In case of `income_composition_of_resources`, we will leave the feature as it is. Our cluster method will probably not work, because there's not much fluctuaion in the data, so cluster centers will be close to eachother, and by that our method with calculating probability will assign high probability to each cluster:

In [ ]:
experimental_train_data = experimental_train_data.join(CS_data)
experimental_train_data.head(5)

### 4.2.2 Plotting the Data

In this part, we will try to visulise the data, in order to get better understanding about the relations of the data, and try to understand what features have negative or positive impact on life expectancy. One thing that we know, that `status` column highly impact the value of the life_expectancy. But first let's check if our created features increased the values of correlation.

In [ ]:
def find_correlated_features(threshold: int, columns = None) -> pd.DataFrame:
    """
    Function that checks highly correlated features
    :param threshold: threshold of correlation. Only columns with higher or equal correlation with threshold will be displayed.
    :param columns: columns for which we want to check the correlation. (Optional argument)
    :return: DataFrame with correlated pairs
    """
    correlated_pairs = []
    if columns is not None:
        data = experimental_train_data[columns]
    else:  
        data = experimental_train_data.select_dtypes(np.number)
    for i in range(len(data.columns)):
        for j in range (i+1, len(data.columns)):
            col1 = data.columns[i]
            col2 = data.columns[j]
            corr_value = data[[col1, col2]].corr().iloc[1].values[0]
            if abs(corr_value) >= threshold:
                correlated_pairs.append((col1, col2, corr_value.round(3)))
    return pd.DataFrame(correlated_pairs, columns=["Feature_1","Feature_2","Corr"]).pivot_table(
        values="Corr",
        index="Feature_1",
        columns="Feature_2"
    )
    

In [ ]:
keywords = ['pow3','life','log_','adult','infant','percentage','measles','under-','hiv/','popul','hepati','poli','dipht','thin','proba','gdp']
cols = [col for col in experimental_train_data.columns if any(keyword in col for keyword in keywords)]
plt.figure(figsize=(20,8))
sns.heatmap(
    find_correlated_features(0, columns=cols),
    vmin=-1,
    vmax=1,
    annot=True
)

Based on that plot we can say:
* In terms of `log` transformation just `log_adult_mortality` did not increase the correlation between feature and life_expectancy.
* In terms of `cubic` transformation all of our features have increased correlation between life_expectancy and particular feature.
* In terms of `KMeans` transformation two clusters should be enough. However, features connected with thinnes are highly correlated with each other, so we will drop one of this feature
* `GDP` and `Percentage_expenditure` are also highly correlated with eachother, so one of those features will have to be dropped.

Now lets dive into visualisation. First let's check columns that are connected with immunization and sicknesses.

In [ ]:
sickness_features = ['measles','diphtheria','hepatitis_b','polio', 'hiv/aids']
for i,col in enumerate(sickness_features):
    sns.jointplot(
        data=experimental_train_data,
        x='life_expectancy',
        y=col,
        kind="reg",
        line_kws={'color':'red'}
    )
plt.tight_layout()


Based on that we can say, that if a great part of country population is immunized (by vaccination or other forms) to a particular disease, life expectancy of that country can increase dramatically. Now let's check information about general condition of the countries, and how they impact the life expectancy.

In [ ]:
condition_features = ['gdp','income_composition_of_resources']
fig, ax = plt.subplots(1,3,figsize=(12,8))
for i in range(len(condition_features)+1):
    if i == 2:
        sns.scatterplot(
            data=experimental_train_data,
            x=condition_features[0],
            y=condition_features[1],
            ax=ax[i],
            hue='status'
        )
    else:
        sns.scatterplot(
            data=experimental_train_data,
            x='life_expectancy',
            y=condition_features[i],
            ax=ax[i],
            hue='status'
        )
plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.suptitle("General condition of country vs life expectancy", fontsize=20);


As our logic says, the richer / better developed country is, the longer we can expect people to live. 

In [ ]:
death_features = ['adult_mortality','infant_deaths','under-five_deaths']
fig, ax = plt.subplots(2,3,figsize=(12,5))
for i in range(len(death_features)+3):
    
    if i == 3:
        sns.scatterplot(
            data=experimental_train_data,
            x=death_features[0],
            y=death_features[1],
            ax=ax[1,0],
            hue='status'
        )
    elif i == 4:
        sns.scatterplot(
            data=experimental_train_data,
            x=death_features[0],
            y=death_features[2],
            ax=ax[1,1],
            hue='status'
        )
    elif i == 5:
        sns.scatterplot(
            data=experimental_train_data,
            x=death_features[1],
            y=death_features[2],
            ax=ax[1,2],
            hue='status'
        )
    else:
        sns.scatterplot(
            data=experimental_train_data,
            x=death_features[i],
            y='life_expectancy',
            ax=ax[0,i],
            hue='status'
        )
plt.tight_layout()
   

As we see, there's some correlation between our data. For example:
* `infant_deaths` and `under-five_deaths` seems to be almost linear, this means, that they have almost the same values. So one of this feature can be dropped.
* `adult mortality` and `life expectancy` appear to be strongly negatively correlated: as the values of adult mortality increase, life expectancy tends to decrease.

The last features to visualise are: `alcohol`, `year`, `schooling` and `bmi`.

In [ ]:
year_data = experimental_train_data.groupby(
    by=['status','year']
).agg(
    {'life_expectancy':'mean'}
)

plt.figure(figsize=(12,8))
sns.lineplot(
    data=year_data,
    y='life_expectancy',
    x='year',
    hue='status'
)
plt.tight_layout()
plt.title("Mean life expectancy over the years", fontsize=15);
plt.grid()

As we can see, over the years, the mean life expectancy has been increasing for both developed and developing countries. On average, life expectancy has risen by about 5 years over a span of 15 years. Now lets check the alcohol variable.

In [ ]:
plt.figure(figsize=(12,8))
sns.regplot(
    data=experimental_train_data,
    x='alcohol',
    y='life_expectancy',
    line_kws={'color':'crimson'},
    ci=None
);
plt.title("Alcohol vs Life Expectancy");

And here's something interesting. Based on this plot the more particular drinks alcohol, we expect people to live longer. This statement seems not to be true, since alcohol is not good for our health. There's probably another varable that influences this behaviour. Let's look at the statistics groupped by status column.

In [ ]:
experimental_train_data.groupby(
    by='status'
).agg(
    {'alcohol':['mean','median']}
).reset_index()

Developed countries seems to drink a lot more alcohol than developing countries, but it's not a factor that increases your life expectancy - it's just a result of living in a developed country, where you can buy more easily a lot of alcohol. Lets check the correlation for both those groups.

In [ ]:
developed = experimental_train_data[experimental_train_data['status']=='Developed']
developing = experimental_train_data[experimental_train_data['status']=='Developing']

developed_corr = developed[['life_expectancy','alcohol']].corr()['alcohol'].values[0].round(3)
developing_corr = developing[['life_expectancy','alcohol']].corr()['alcohol'].values[0].round(3)

print(f"Correlation between alcohol and life expectancy for developed countries: {developed_corr}")
print(f"Correlation between alcohol and life expectancy for developing countries: {developing_corr}")



We can see that, in fact the more highly developed countries drink, the lower is the life expectancy. However, in developing countries it seems that drinking alcohol can extend your life (joking)!

Let's build some basic OLS model to further analyse this variable:

In [ ]:
import statsmodels.formula.api as smf

model_data = experimental_train_data.copy()
model_data['status_encoded'] = model_data['status'].apply(lambda x: 1 if x == 'Developed' else 0)
model = smf.ols('life_expectancy ~ alcohol + status_encoded * alcohol', data=model_data).fit()

In [ ]:
print(model.summary())

In terms of alcohol, developed countries that drink much alcohol seems to have lower life expectancy. (Real Inpact =  $0.573 - 0.961 = -0.387$). Based od that, we will keep this variable for now. Now we're left with `schooling` and `bmi` variables.

In [ ]:
sch_bmi = ['schooling','bmi']
for i, col in enumerate(sch_bmi):
    sns.jointplot(
        data=experimental_train_data,
        x=col,
        y="life_expectancy",
        hue="status"
    )

Schooling seems to have significat influence on life expectancy. Better access to education seems to rise the level of life expectancy. Higher BMI seems to correlate with higher life expectancy but theres probably the same situation as in alcohol variable. Lets see how they perform on a simple OLS model.

In [ ]:
model = smf.ols('life_expectancy ~ schooling + status_encoded * schooling + bmi + status_encoded*bmi', data=model_data).fit()
print(model.summary())

The Real impact on Developed countries for BMI is: $0.122 - 0.146 = -0.024$, and for Schooling: $1.661 - 0.6582 = 1.002$. The last step is to check correlation for all of our variables.

In [ ]:
plt.figure(figsize=(20,8))
sns.heatmap(
    find_correlated_features(0.6),
    vmin=-1,
    vmax=1,
    annot=True
)

# 5. Building the Pipelines



In this part, we will try to build a pipeline that will prepare our data for modeling. To summarise, we will:

* Delete: `percentage_expenditure`, `infant_deaths`, `thinness_5-9_years`, `population`
* Perform `Log` Transformation on: `hiv/aids`, `gdp`, `measles`,
* Perform `Cubic` Transformation on: `diphtheria`, `hepatitis_b`, `polio`,
* Perform `KMeans` and `RBF` Tranformation on : `thinness_1-19_years`,
* OneHot encode categorical variables,
* Standarize necessary columns,
* Impute missing values using `KMeans` algorithm.
* Add an `IsolationForest` algorithm to remove outliers.

In [ ]:
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest

## 5.1 Logarithimic Pipeline

In [ ]:
LOG_FEATURES = ["hiv/aids", "gdp", "measles", "under-five_deaths"]

def replace_zeros_with_min(X):
    """
    Function that replaces 0 with very small values in order to gain the possibility to log them.
    """
    return np.where(X==0, 1e-9, X)

replacing_zeros_transformer = FunctionTransformer(replace_zeros_with_min, validate=False, feature_names_out="one-to-one")
log_operation_transformer = FunctionTransformer(np.log, inverse_func=np.exp, feature_names_out="one-to-one")


log_pipeline = make_pipeline(
    KNNImputer(n_neighbors=7),
    replacing_zeros_transformer,
    log_operation_transformer,
    StandardScaler()
)



## 5.2 Cubic Pipeline

In [ ]:
CUBIC_FEATURES = ['diphtheria', 'hepatitis_b', 'polio']

def cube(X):
    return np.power(X,3)


cubic_transformer = FunctionTransformer(cube, inverse_func=np.cbrt, feature_names_out='one-to-one')

cubic_pipeline = make_pipeline(
    KNNImputer(n_neighbors=7),
    cubic_transformer,
    StandardScaler()
)

## 5.3 Cluster Pipeline

In [ ]:
CLUSTER_FEATURES = ['thinness_1-19_years']
cluster_pipeline = make_pipeline(
    KNNImputer(n_neighbors=7),
    ClusterSimilarity(n_clusters=3)
)


## 5.4 Categorical Features Pipeline

In [ ]:
CAT_FEATURES = ["year","status"]
cat_pipeline = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(sparse_output=False, handle_unknown="ignore")
)

## 5.5 Default Numeric Pipeline - Parametric

In [ ]:
default_num_pipeline = make_pipeline(
    KNNImputer(n_neighbors=7),
    StandardScaler()
)

## 5.6 Default Numeric Pipeline - Non Parametric

In [ ]:
default_num_pipeline_tree = make_pipeline(
    KNNImputer(n_neighbors=7)
)

## 5.7 Target Pipeline

In [ ]:
TARGET = ['life_expectancy']

tar_pipeline = make_pipeline(
    KNNImputer(n_neighbors=7)
)

## 5.8 ColumnTransformer - Parametric

In [ ]:
REST_FEATURES = ["adult_mortality", "alcohol", "total_expenditure", "income_composition_of_resources","schooling"]

preprocessing_pipeline_parametric = ColumnTransformer([
    ("log", log_pipeline, LOG_FEATURES),
    ("cub", cubic_pipeline, CUBIC_FEATURES),
    ("clu", cluster_pipeline, CLUSTER_FEATURES),
    ("cat", cat_pipeline, CAT_FEATURES),
    ("tar", tar_pipeline, TARGET),
    ("res", default_num_pipeline, REST_FEATURES)
    
])


## 5.9 ColumnTransformer - Non parametric

In [ ]:
preprocessing_pipeline_non_param = ColumnTransformer([
    ("tar", tar_pipeline, TARGET),
    ("cat", cat_pipeline, CAT_FEATURES),
    ("clu", cluster_pipeline, CLUSTER_FEATURES),
    ("res", default_num_pipeline_tree, LOG_FEATURES + CUBIC_FEATURES + REST_FEATURES)
])


## 5.10 IsolationForest Class

We have to create custom class in order to get acces to the tranform method, which will delete the rows, which are considered as outliers.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import IsolationForest

class OutlierRemoval(BaseEstimator, TransformerMixin):
    def __init__(self, contamination=0.1, random_state=None):
        self.contamination = contamination
        self.random_state = random_state

    def fit(self, X, y=None):
        self.isolation_ = IsolationForest(contamination=self.contamination, random_state=self.random_state)
        self.isolation_.fit(X)
        return self

    def transform(self, X):
        is_inliner_mask = self.isolation_.predict(X) == 1
        return X[is_inliner_mask]

    def fit_transform(self, X, y=None):
        self.fit(X)
        return self.transform(X)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return input_features
        return input_features


In [ ]:
full_pipeline_param = Pipeline([
    ("preprocessing", preprocessing_pipeline_parametric),
    ("outlier_removal", OutlierRemoval(contamination=0.1, random_state=42))
])

full_pipeline_non_param = Pipeline([
    ("preprocessing", preprocessing_pipeline_non_param),
    ("outlier_removal", OutlierRemoval(contamination=0.1, random_state=42))
])

In [ ]:
full_pipeline_param

## 5.11 Transforming the Data

In [ ]:
pd.DataFrame(
    full_pipeline_param.fit_transform(strat_train_set),
    columns=PREPROCESING_PARAM.get_feature_names_out()
)

# 6. Modeling the Data

In this part we will build some models in order to predict the life expectancy. But first let's divide our data.

In [ ]:

PREPROCESSING_PARAM = full_pipeline_param.fit(strat_train_set)
PREPROCESSING_NON_PARAM = full_pipeline_non_param.fit(strat_train_set)

ml_data_param_train = pd.DataFrame(
    PREPROCESSING_PARAM.transform(strat_train_set),
    columns=PREPROCESSING_PARAM.get_feature_names_out()
)

ml_data_non_param_train = pd.DataFrame(
    PREPROCESSING_NON_PARAM.transform(strat_train_set),
    columns=PREPROCESSING_NON_PARAM.get_feature_names_out()
)


ml_data_param_test = pd.DataFrame(
    PREPROCESSING_PARAM.transform(strat_test_set),
    columns=PREPROCESSING_PARAM.get_feature_names_out()
)

ml_data_non_param_test = pd.DataFrame(
    PREPROCESSING_NON_PARAM.transform(strat_test_set),
    columns=PREPROCESSING_NON_PARAM.get_feature_names_out()
)


y = 'tar__life_expectancy'

#Train set

X_train_param = ml_data_param_train.drop(y, axis=1)
y_train_param = ml_data_param_train[y]

X_train_non_param = ml_data_non_param_train.drop(y, axis=1)
y_train_non_param = ml_data_non_param_train[y]

#Test set

X_test_param = ml_data_param_test.drop(y, axis=1)
y_test_param = ml_data_param_test[y]

X_test_non_param = ml_data_non_param_test.drop(y, axis=1)
y_test_non_param = ml_data_non_param_test[y]




## 6.1 Lasso Regression

First model that we will build is the `Lasso Regression` since I assume that not every feature is very important in general. Lasso Regression modify our cost function: $MSE(\theta) = \frac{1}{n}\sum\limits_{i=1}^n(y_i - \hat{y_i})^2$ by adding a regularisation part: $2\alpha\sum\limits_{i=1}^n|\theta_i|$. By adding that part, our model will not only try to minimalise `MSE` but also `minimalise weights` in our model (it can also set some weights to 0), so it can be used to select features that important for our modeling. We will also perform `GridSearch` in order to find the best value of `alpha` hiperparameter.

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV

In [ ]:
lasso = Lasso(random_state=42)
params = {
    'alpha':[0.001, 0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    lasso,
    params,
    n_jobs=-1,
    cv=5,
    scoring='neg_root_mean_squared_error'
)
grid_search.fit(X_train_param, y_train_param)
print(f"Best alpha value: {grid_search.best_params_}")
print(f"Best RMSE score: {-grid_search.best_score_}")

In [ ]:
result_data = pd.DataFrame(grid_search.cv_results_)
cols = [col for col in result_data.columns if 'split' in col or 'params' in col]
result_data = result_data[cols]
result_data['params'] = result_data['params'].apply(lambda x: x.get('alpha'))
result_data = abs(result_data)
sns.heatmap(result_data.pivot_table(
    index='params',
    columns=None
),
            annot=True
)
plt.title("Params vs RMSE");


It's always a good idea to add some form of regularisation to our models to prevent regularisation. As we see, after we try to give higher values of `alpha` in our model, our `MSE` rises, so it's better to stick with lower values of the penalty. As we previously mentioned, Lasso Regression can be used to `feature selection`, lets see the weights of our features.

In [ ]:
best_lasso = grid_search.best_estimator_
coefs = best_lasso.coef_
features = best_lasso.feature_names_in_

#Sorting

sorted_indices = np.argsort(coefs)
feature_names_sorted = features[sorted_indices]
coef_sorted = coefs[sorted_indices]

#Plot Creation

plt.figure(figsize=(17,8))
bars = plt.bar(
    feature_names_sorted,
    coef_sorted
)
ax = plt.gca()
for tick, coef in zip(ax.get_xticklabels(), coef_sorted):
    if coef == 0 :
        tick.set_color("red")
plt.tight_layout()
plt.title("Weights of Lasso Regression", fontsize=15);
plt.xticks(rotation=60, ha='right');
plt.ylabel("Weights", fontsize=12);

Features markeed as red, are features that our model classified as not important. The biggest influence on life expectancy have feature `hiv/aids` and `schooling`, which makes sense based on our previous analysis. We will save the features with weights = 0 to remove them from future modeling process. But now, lets see how the model will perform on validation data.

In [ ]:
from sklearn.metrics import root_mean_squared_error

preds_test = best_lasso.predict(X_test_param)
preds_train = best_lasso.predict(X_train_param)
train_score = np.round(best_lasso.score(X_train_param, y_train_param),3)
test_score = np.round(best_lasso.score(X_test_param, y_test_param),3)

train_rmse = root_mean_squared_error(y_train_param, preds_train).round(3)
test_rmse = root_mean_squared_error(y_test_param, preds_test).round(3)

print(f"Train score: {train_score}, with RMSE = {train_rmse}")
print(f"Test score: {test_score}, with RMSE = {test_rmse}")

LOW_WEIGHTS_ATTRIBS_LASSO = list(best_lasso.feature_names_in_[best_lasso.coef_ == 0])


## 6.2 Elastic Net Regression

Another great model is called `Elastic Net`. It's like a middle man between `Ridge Regression` and `Lasso Regression`. In this case, regularization element is a weighted sum of both Lasso and Ridge Regression. Our cost function looks like this:

$J(\theta) = MSE(\theta) + r (2\alpha\sum\limits_{1=i}^n|\theta_i|) + (1-r)(\frac{a}{m}\sum\limits_{i=1}^n\theta_i^2)$

One thing to notice, if $r=1$ then we will perform `Lasso Regression`, if $r=0$ then we will perform Ridge Regression.

In [ ]:
from sklearn.linear_model import ElasticNet

params = {
    'alpha':[0.001, 0.01, 1, 10],
    'l1_ratio':[0.1, 0.3, 0.5, 0.7, 0.9, 1]
}

model = ElasticNet(random_state=42)
grid_search = GridSearchCV(model, params, cv=5, scoring='neg_root_mean_squared_error')
grid_search.fit(X_train_param, y_train_param)
print(f"Best params are: {grid_search.best_params_}")
print(f"Best score is: {np.round(-grid_search.best_score_,3)}")


As we see there's nothing to compare, since `l1_ratio = 1`. This means, that ElasticNet are performing in fact `Lasso Regression`. We will switch to tree like models.

## 6.4 Random Forest Regressor

In this part, we will try to build `Random Forest` model. It will contain a lot of `Decision Trees` from which we will take outputs and average them in order to get final prediction. One of the biggest advantages of them is that they don't have any assumptions for out data, making them so powerful. However since they are so powerful they have significant variance, meaning that small difference in hiperparameters can give us very different models. In other fact Random Forest models are very easy to overfit. Lets try to build one.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

params = {
    'n_estimators':[150,200],
    'max_leaf_nodes':[10,15],
    'min_samples_split':[10,15],
    'min_samples_leaf':[5,10,15],
    'max_depth':[5,6,7]
}

model = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    model,
    params,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

grid_search.fit(X_train_non_param, y_train_non_param)
print(f"Best estimator: {grid_search.best_estimator_}")
print(f"Best score: {np.round(-grid_search.best_score_,3)}")

In [ ]:
best_random_forest = grid_search.best_estimator_

preds_train = best_random_forest.predict(X_train_non_param)
preds_test = best_random_forest.predict(X_test_non_param)
score_train = best_random_forest.score(X_train_non_param, y_train_non_param)
score_test = best_random_forest.score(X_test_non_param, y_test_non_param)
train_rmse = root_mean_squared_error(y_train_non_param, preds_train)
test_rmse = root_mean_squared_error(y_test_non_param, preds_test)

print(f"Score for train data: {np.round(score_train,3)}, with RMSE = {train_rmse.round(3)}")
print(f"Score for test data: {np.round(score_test,3)}, with RMSE = {test_rmse.round(3)}")



As we see there's huge difference for RMSE in comparision with Lasso Regression. Lets look at the feature importances.

In [ ]:
importances = best_random_forest.feature_importances_
features = best_random_forest.feature_names_in_
sorted_indices = np.argsort(importances)
sorted_features = features[sorted_indices]
sorted_importances = importances[sorted_indices]

plt.figure(figsize=(12,8))

plt.bar(
    sorted_features,
    sorted_importances
);
plt.tight_layout()
plt.xticks(rotation=45, ha="right");

As we see there's a lot of features that our model did not use. Let's try deleting them from our model, and let's try train that model again.

In [ ]:
IMPORTANT_FEATURES_TREE = features[importances.round(3) != 0]

X_train_non_param_imp = X_train_non_param[IMPORTANT_FEATURES_TREE]
X_test_non_param_imp = X_test_non_param[IMPORTANT_FEATURES_TREE]


from sklearn.ensemble import RandomForestRegressor

params = {
    'n_estimators':[150,200,250],
    'max_leaf_nodes':[10,15],
    'min_samples_split':[10,15],
    'min_samples_leaf':[5,10,15],
    'max_depth':[5,6,7,8]
}

model = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    model,
    params,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

grid_search.fit(X_train_non_param_imp,y_train_non_param)
print(f"Best estimator: {grid_search.best_estimator_}")
print(f"Best score: {-grid_search.best_score_}")


best_random_forest = grid_search.best_estimator_

preds_train = best_random_forest.predict(X_train_non_param_imp)
preds_test = best_random_forest.predict(X_test_non_param_imp)
score_train = best_random_forest.score(X_train_non_param_imp, y_train_non_param)
score_test = best_random_forest.score(X_test_non_param_imp, y_test_non_param_imp)
train_rmse = root_mean_squared_error(y_train_non_param, preds_train)
test_rmse = root_mean_squared_error(y_test_non_param_imp, preds_test)

print(f"Score for train data: {np.round(score_train,3)}, with RMSE = {train_rmse.round(3)}")
print(f"Score for test data: {np.round(score_test,3)}, with RMSE = {test_rmse.round(3)}")



As we see we got almost the same results as before, so this means, that we don't need this much features. Lets try to create a little bit more complicated models.

## 6.5 AdaBoost Regressor

New predictor can improve his predecessor by giving more atention to the train samples for which previous algorithm was undertrained. By this way, new predictor will give more attention to harder and harder samples. For `weak learner` we will use a single `Decision Tree`.

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor


tree = DecisionTreeRegressor(random_state=42)
ada_reg = AdaBoostRegressor(tree, random_state=42, learning_rate=0.01)

params = {
    'n_estimators':[100,150],
    'loss':['linear', 'square'],
    'estimator__max_depth':[7,8],
    'estimator__max_leaf_nodes':[10,15],
    'estimator__min_samples_split':[10,15],
    'estimator__min_samples_leaf':[10,15],
}

grid_search = GridSearchCV(
    ada_reg,
    params,
    cv=3,
    scoring="neg_root_mean_squared_error"
)

grid_search.fit(X_train_non_param_imp, y_train_non_param)
print(f"Best estimator: {grid_search.best_estimator_}")

best_ada = grid_search.best_estimator_

preds_train = best_ada.predict(X_train_non_param_imp)
preds_test = best_ada.predict(X_test_non_param_imp)
score_train = best_ada.score(X_train_non_param_imp, y_train_non_param)
score_test = best_ada.score(X_test_non_param_imp, y_test_non_param_imp)
train_rmse = root_mean_squared_error(y_train_non_param, preds_train)
test_rmse = root_mean_squared_error(y_test_non_param_imp, preds_test)

print(f"Score for train data: {np.round(score_train,3)}, with RMSE = {train_rmse.round(3)}")
print(f"Score on test: {np.round(score_test,3)}, with RMSE = {test_rmse.round(3)}")

AdaBoost regressor performed a little bit better than RandomForest model, however we can still see that theres a little bit of overfitting, since our model is performing better on train dataset (but not much). Our model is right now highly regularised by hiperparameters. For now, lets try Gradient Boosting models.

## 6.6 GradientBoostingRegressor

Gradient Tree Boosting models are working similar to AdaBoost models, however they are not updating weights of each sample, but we're trying to fit predictor to the `residual error` commited by predecessor. 

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

params = {
    'n_estimators':[300],
    'min_samples_split':[30,35],
    'min_samples_leaf':[30,35],
    'max_depth':[3,4]
}

gbr = GradientBoostingRegressor(random_state=42, n_iter_no_change=50, learning_rate=0.01)

grid_search = GridSearchCV(
    gbr,
    params,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train_non_param_imp, y_train_non_param)

print(f"Best estimator: {grid_search.best_estimator_}")
print(f"Best score: {-np.round(grid_search.best_score_,3)}")

best_gbr = grid_search.best_estimator_

preds_train = best_gbr.predict(X_train_non_param_imp)
preds_test = best_gbr.predict(X_test_non_param_imp)

score_train = np.round(best_gbr.score(X_train_non_param_imp, y_train_non_param),3)
score_test = np.round(best_gbr.score(X_test_non_param_imp, y_test_non_param_imp),3)

rmse_train = root_mean_squared_error(y_train_non_param, preds_train).round(3)
rmse_val = root_mean_squared_error(y_test_non_param_imp, preds_test).round(3)

print(f"Score on train: {score_train}, with RMSE = {rmse_train}")
print(f"Score on test: {score_test}, with RMSE = {rmse_val}")


Gradient Boosting Model have performed better than our previous models, however we still can see a slight of overfitting. Now let's try `XGBoost`.

## 6.7 XGBoost Regressor

XGBoost Regressor works similar to the Gradient Tree Boosting Regressor, but uses a different optimizing approach. More information are here: https://xgboost.readthedocs.io/en/stable/tutorials/model.html

In [ ]:
import xgboost as xg

model = xg.XGBRegressor(seed=42, validate_parameters = True, subsample=0.5, grow_policy = 'lossguide')

params = {
    'eta':[0.01],
    'max_depth':[4,5],
    'lambda':[0.1, 0.2],
    'gamma':[0.1, 0.2],
    'n_estimators':[250],
    'max_leaves':[20,25]

}

grid_search = GridSearchCV(
    model,
    params,
    cv=3,
    scoring='neg_root_mean_squared_error'
)

grid_search.fit(X_train_non_param_imp, y_train_non_param)

print(f"Best estimator: {grid_search.best_estimator_}")
print(f"Best score: {-np.round(grid_search.best_score_,3)}")

best_xgb = grid_search.best_estimator_

preds_train = best_xgb.predict(X_train_non_param_imp)
preds_test = best_xgb.predict(X_test_non_param_imp)

score_train = np.round(best_xgb.score(X_train_non_param_imp, y_train_non_param),3)
score_test = np.round(best_xgb.score(X_test_non_param_imp, y_test_non_param),3)

rmse_train = root_mean_squared_error(y_train_non_param, preds_train).round(3)
rmse_test = root_mean_squared_error(y_test_non_param, preds_test).round(3)

print(f"Score on train: {score_train}, with RMSE = {rmse_train}")
print(f"Score on test: {score_test}, with RMSE = {rmse_test}")

# 7. Conclusion

As we see our models are slightly overfitted. We could see if there's any differences between our validation and train data, cause as we see theres a little bit of overfitting in almost every model or maybe we could gather some more data in both validation and train data, and then train the models again, or take some more time in order to find the perfect hyperparameters. I think that the best model as it is, is the `Gradient Tree Boosting` since results on train and validation data are most similar to eachother.